# FlyRank Capstone — Leakage-Aware Content Decline Prioritization

## Research question

**For an SEO/content editor, can prior-window and contextual page signals prioritise a review queue that captures a higher share of currently declining pages than a transparent freshness-and-visibility rule?**

> This is **directional decision support**, not an autonomous refresh decision, a causal claim, or a prediction of Google's algorithm. The target is a current-snapshot decline proxy, so all last-30-day, 90-day, and label-derived fields are excluded from model features.

The full written report is in `work/capstone_report.md`. Run the cells below from top to bottom; the analysis script produces reproducible metrics, a ranked queue, permutation importances, and two charts under `work/outputs/`.

## 1. Data and safety contract

The bundled data has 30,000 pseudonymised pages across 32 pseudonymised clients. `client_id` is used only to hold out whole clients for testing; it is not a feature. `trend_direction` defines the label and is never a feature.

The project uses only the **preceding 30-day counts** and stable/contextual signals as predictors. It intentionally excludes every most-recent-30-day field and every 90-day activity total/rate because those overlap the outcome period.

In [1]:
from pathlib import Path
import subprocess
import sys

def locate_repo() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists() and (candidate / "work/scripts/run_capstone.py").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from the repository, then run it again.")

REPO = locate_repo()
print(f"Repository found: {REPO}")

Repository found: C:\Users\Danish Computer\OneDrive\Desktop\flyrankmlprojectclone31_7\flyrankmlproject


## 2. Reproducible analysis

The script uses a client-held-out split, a transparent baseline, a random-forest model with missing-data safeguards, and evaluation at the decision-relevant queue sizes (50 and 100 pages).

In [2]:
subprocess.run(
    [sys.executable, str(REPO / "work/scripts/run_capstone.py")],
    cwd=REPO,
    check=True,
)

CompletedProcess(args=['C:\\Program Files\\Python312\\python.exe', 'C:\\Users\\Danish Computer\\OneDrive\\Desktop\\flyrankmlprojectclone31_7\\flyrankmlproject\\work\\scripts\\run_capstone.py'], returncode=0)

## 3. Evaluation evidence

**Primary metric:** Precision@50 on held-out clients, shown alongside the held-out decline base rate. The baseline and the model are compared on exactly the same pages.

Read the table generated by the fresh run below; do not type estimated values into the report.

In [3]:
from IPython.display import Markdown, display
print((REPO / "work/outputs/capstone_metrics.md").read_text())

# Capstone run metrics

| Measure | Value |
|---|---:|
| Rows | 30,000 |
| Pseudonymized clients | 32 |
| Train / held-out rows | 23,837 / 6,163 |
| Train / held-out clients | 25 / 7 |
| Held-out decline base rate | 0.511 |
| Model ROC-AUC | 0.659 |
| Model average precision | 0.616 |
| Model Precision@50 | 0.540 |
| Baseline Precision@50 | 0.340 |
| Model Precision@100 | 0.460 |
| Baseline Precision@100 | 0.340 |

**Interpretation boundary.** This run prioritizes pages that show an observed decline in the
current snapshot using prior-window and contextual signals. It is directional decision support,
not a causal estimate, a forecast beyond this snapshot, or a claim about Google's algorithm.



## 4. Interpretation and recommendation

Permutation importance reports which *feature columns* most helped distinguish the target on the held-out clients. It is not a causal explanation. The operational output is a review queue; the editor must investigate each page before changing it.

In [4]:
import pandas as pd
print("Public-safe model-artifact preview (page-level identifiers and query text withheld)")
feature_path = REPO / "work" / "outputs" / "capstone_feature_importance.csv"
if feature_path.exists():
    public_feature_importance = pd.read_csv(feature_path)
    display(public_feature_importance.head(10))
else:
    print("Feature-importance artifact will be available after the analysis run.")


Public-safe model-artifact preview (page-level identifiers and query text withheld)


,feature,importance_mean_auc_drop,importance_std
0,log_impressions_prev_30d,0.140583,0.004687
1,log_clicks_prev_30d,0.010662,0.000644
2,content_age_days,0.009116,0.003321
3,log_sessions_prev_30d,0.005777,0.001210
4,age_tier,0.000607,0.002521
5,search_volume,0.000209,0.001151
6,content_type,0.000000,0.000000
7,competition,-0.000546,0.000198
8,cpc,-0.000787,0.000418
9,char_count,-0.001021,0.001746


## 5. Limitations and honest claims

1. The label captures an observed decline **inside the current snapshot**; this starter-data version cannot claim to forecast beyond the snapshot.
2. The ranking identifies pages for human review, not a guaranteed refresh outcome.
3. The model does not establish that an editor action will cause traffic to recover. A future warehouse-based extension should define a forward outcome window and assess the result with a time-aware split.
4. No client-identifying information, raw data, or private query text is included in the notebook or the report.

**Next writing step:** after running the notebook, paste the fresh metrics into `work/capstone_report.md`, explain the top importances in plain language, and add a short false-positive/false-negative review before publishing the report.

## 7. ML-12 storytelling pack: demo outline and shareable cuts

### Five-minute demo outline — *Leakage-Aware Content Refresh Prioritization*

**0:00–0:30 | The decision problem.** Content teams rarely have the capacity to refresh every page at once. This case study asks whether public-safe, anonymized prior-window and contextual page signals can help an editor decide which pages deserve a **human review first** when observed decline has occurred.

**0:30–1:25 | Data and label.** I used the FlyRank ML Internship's anonymized content-page release. The analysis keeps the decision time-safe: the label is an observed decline in the current snapshot, while model features are restricted to prior-window or static contextual signals. The notebook does not show client names, page URLs, query text, or page-level identifiers.

**1:25–2:20 | Method.** I compared a transparent hand-built baseline with a random-forest ranking model. Clients were held out as whole groups so that the evaluation asks whether the ranking approach transfers to unseen clients, and a feature-leakage audit removed current-window or label-proxy information.

**2:20–3:15 | One chart and one result.** Show the held-out **Precision@50** chart. On the same seven held-out clients, the model achieved Precision@50 of **0.540** (27 observed-decline pages among the top 50) compared with **0.340** for the baseline (17 of 50).

**3:15–4:05 | Honest interpretation.** This is not proof that a refresh causes recovery, and it is not an automated publishing decision. The outcome is an observed decline inside the available snapshot; the held-out decline rate is already high, so performance should be interpreted as a ranked review queue rather than a causal recommendation.

**4:05–5:00 | Recommendation.** Use the model's top-ranked pages as an editorial-review inbox. An editor should inspect content quality, search intent, freshness, and business context before deciding whether to refresh, consolidate, redirect, or take no action; track follow-up outcomes and re-evaluate the queue before operational use.

### Shareable methodology post

I built a leakage-aware content-refresh prioritization workflow on the anonymized FlyRank ML Internship dataset. I framed the work as a human-review ranking problem, used time-safe prior-window and contextual features, held out whole clients, and compared a transparent baseline with a random-forest model using Precision@50. On seven held-out clients, the model reached 0.540 Precision@50 versus 0.340 for the baseline, but the result is deliberately framed as decision support—not evidence that refreshing content causes recovery. The project includes the evaluation chart, limitations, and a public-safe editorial action playbook.

### Employer-facing summary

I built a reproducible, leakage-aware content-refresh prioritization system using the anonymized FlyRank ML Internship content-page dataset. I evaluated a random-forest ranking model against a transparent baseline on unseen clients; in the top 50 candidates, it identified 27 observed-decline pages (Precision@50 = 0.540) versus 17 (0.340) for the baseline. The project turns model scores into human-review recommendations while documenting important limitations, including that observed decline labels do not establish causality.
